# Predicting Fraud Risk Through Authentication and Transaction Behavior in Digital Banking

**Project focus:** Analyze how authentication methods, transaction behavior, and digital banking activity patterns relate to fraudulent transactions.

In [ ]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.3f}".format)

In [ ]:
# Load the dataset
# Update the file path if your CSV is saved in a different folder.

df = pd.read_csv("banking_transactions.csv")

print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Basic dataset information

print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Clean the dataset

# Convert fraud_flag to integer if it is currently True/False
df["fraud_flag"] = df["fraud_flag"].astype(int)

# Drop transaction_id because it is only an identifier and should not be used for prediction
if "transaction_id" in df.columns:
    df_model = df.drop(columns=["transaction_id"])
else:
    df_model = df.copy()

print("Cleaned dataset shape:", df_model.shape)
df_model.head()

## Exploratory Data Analysis

This section explores fraud distribution, authentication methods, login attempts, suspicious IP activity, device risk, and other digital banking behaviors.

In [ ]:
# Fraud distribution

fraud_counts = df_model["fraud_flag"].value_counts().sort_index()
fraud_rate = df_model["fraud_flag"].mean()

print("Fraud counts:")
print(fraud_counts)
print(f"\nFraud rate: {fraud_rate:.2%}")

plt.figure(figsize=(6,4))
fraud_counts.plot(kind="bar")
plt.title("Fraud vs Non-Fraud Transactions")
plt.xlabel("Fraud Flag (0 = Non-Fraud, 1 = Fraud)")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Fraud rate by authentication type

auth_fraud = (
    df_model.groupby("authentication_type")["fraud_flag"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

auth_fraud["fraud_rate_percent"] = auth_fraud["mean"] * 100
display(auth_fraud)

plt.figure(figsize=(8,4))
auth_fraud["fraud_rate_percent"].plot(kind="bar")
plt.title("Fraud Rate by Authentication Type")
plt.xlabel("Authentication Type")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
# Fraud rate by payment channel

channel_fraud = (
    df_model.groupby("payment_channel")["fraud_flag"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

channel_fraud["fraud_rate_percent"] = channel_fraud["mean"] * 100
display(channel_fraud)

plt.figure(figsize=(8,4))
channel_fraud["fraud_rate_percent"].plot(kind="bar")
plt.title("Fraud Rate by Payment Channel")
plt.xlabel("Payment Channel")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
# Suspicious IP and international transaction fraud rates

binary_features = ["suspicious_ip_flag", "international_transaction_flag", "card_present_flag"]

for col in binary_features:
    summary = df_model.groupby(col)["fraud_flag"].agg(["count", "mean"])
    summary["fraud_rate_percent"] = summary["mean"] * 100
    print(f"\nFraud rate by {col}:")
    display(summary)

    plt.figure(figsize=(6,4))
    summary["fraud_rate_percent"].plot(kind="bar")
    plt.title(f"Fraud Rate by {col}")
    plt.xlabel(col)
    plt.ylabel("Fraud Rate (%)")
    plt.xticks(rotation=0)
    plt.show()

In [ ]:
# Compare numeric variables by fraud status

numeric_cols = df_model.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != "fraud_flag"]

fraud_numeric_summary = df_model.groupby("fraud_flag")[numeric_cols].mean().T
fraud_numeric_summary.columns = ["Non-Fraud Average", "Fraud Average"]
fraud_numeric_summary["Difference"] = fraud_numeric_summary["Fraud Average"] - fraud_numeric_summary["Non-Fraud Average"]
fraud_numeric_summary = fraud_numeric_summary.sort_values("Difference", ascending=False)

display(fraud_numeric_summary)

In [ ]:
# Boxplots for important behavioral/risk variables

important_numeric = [
    "login_attempts",
    "device_risk_score",
    "transaction_velocity_score",
    "anomaly_score",
    "geo_distance_km",
    "session_duration_minutes",
    "failed_transactions_last_30d"
]

for col in important_numeric:
    if col in df_model.columns:
        plt.figure(figsize=(6,4))
        df_model.boxplot(column=col, by="fraud_flag")
        plt.title(f"{col} by Fraud Status")
        plt.suptitle("")
        plt.xlabel("Fraud Flag (0 = Non-Fraud, 1 = Fraud)")
        plt.ylabel(col)
        plt.show()

In [ ]:
# Correlation heatmap for numeric variables

corr = df_model[numeric_cols + ["fraud_flag"]].corr()

plt.figure(figsize=(12,8))
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# Show variables most correlated with fraud
fraud_corr = corr["fraud_flag"].drop("fraud_flag").sort_values(key=abs, ascending=False)
display(fraud_corr)

## Machine Learning Models

The goal is to predict whether a transaction is fraudulent. Because fraud data can be imbalanced, the notebook evaluates models using precision, recall, F1-score, ROC-AUC, and confusion matrices instead of relying only on accuracy.

In [ ]:
# Prepare features and target

X = df_model.drop(columns=["fraud_flag"])
y = df_model["fraud_flag"]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training fraud rate:", y_train.mean())
print("Testing fraud rate:", y_test.mean())

In [ ]:
# Preprocessing steps

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

In [ ]:
# Define models

log_reg = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

rf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        class_weight="balanced",
        random_state=42
    ))
])

# scale_pos_weight helps XGBoost handle class imbalance
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
scale_pos_weight = negative_count / positive_count if positive_count > 0 else 1

xgb = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=42
    ))
])

models = {
    "Logistic Regression": log_reg,
    "Random Forest": rf,
    "XGBoost": xgb
}

In [ ]:
# Train and evaluate models

results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

    print(f"\n{name} Classification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Non-Fraud", "Fraud"])
    disp.plot()
    plt.title(f"{name} Confusion Matrix")
    plt.show()

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
display(results_df)

In [ ]:
# Compare model performance visually

metrics = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]

for metric in metrics:
    plt.figure(figsize=(8,4))
    results_df.set_index("Model")[metric].plot(kind="bar")
    plt.title(f"Model Comparison: {metric}")
    plt.ylabel(metric)
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha="right")
    plt.show()

In [ ]:
# ROC curves

plt.figure(figsize=(7,5))

for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    auc_score = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

## Feature Importance

Feature importance helps explain which variables were most useful for predicting fraud risk. This is especially important because the project is not only about model accuracy, but also about understanding the business meaning behind fraud risk.

In [ ]:
# Get feature names after preprocessing

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("Best model based on ROC-AUC:", best_model_name)

feature_names = best_model.named_steps["preprocess"].get_feature_names_out()

# Feature importance works directly for Random Forest and XGBoost
if best_model_name in ["Random Forest", "XGBoost"]:
    importances = best_model.named_steps["model"].feature_importances_
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False)

# Logistic Regression uses coefficient size as importance
else:
    coefficients = best_model.named_steps["model"].coef_[0]
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": np.abs(coefficients)
    }).sort_values("Importance", ascending=False)

display(importance_df.head(15))

plt.figure(figsize=(9,5))
importance_df.head(15).sort_values("Importance").plot(
    x="Feature",
    y="Importance",
    kind="barh",
    legend=False
)
plt.title(f"Top 15 Feature Importances - {best_model_name}")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## Business-Focused Result Summary

Use the cells below after running the notebook to help write the Analysis and Conclusion sections of the white paper.

In [ ]:
# Automatically generate a short result summary

best = results_df.iloc[0]

print("RESULT SUMMARY")
print("--------------")
print(f"The best-performing model based on ROC-AUC was {best['Model']}.")
print(f"ROC-AUC: {best['ROC-AUC']:.3f}")
print(f"Precision: {best['Precision']:.3f}")
print(f"Recall: {best['Recall']:.3f}")
print(f"F1-score: {best['F1-score']:.3f}")

print("\nTop fraud-related correlations:")
display(fraud_corr.head(10))

print("\nTop model features:")
display(importance_df.head(10))

In [ ]:
# Save key outputs for your paper

results_df.to_csv("model_results_summary.csv", index=False)
fraud_corr.to_csv("fraud_correlation_results.csv")
importance_df.to_csv("feature_importance_results.csv", index=False)

print("Saved result files:")
print("- model_results_summary.csv")
print("- fraud_correlation_results.csv")
print("- feature_importance_results.csv")

## Suggested Interpretation Guide

After running the notebook, use the model comparison table, confusion matrices, ROC curves, and feature importance chart to write your results.

A strong final interpretation can focus on:
- Which authentication method had the highest or lowest fraud rate.
- Whether login attempts, suspicious IP flags, device risk scores, transaction velocity, or anomaly scores were important fraud indicators.
- Which model performed best and why.
- Why recall matters in fraud detection because missed fraud cases may create financial loss.
- Why precision also matters because false positives may inconvenience legitimate customers.